In [1]:
import pandas as std
import sqlite3
import os
print(os.getcwd())
file=std.read_excel("companyfile.xlsx")                    
file


C:\Users\tusha\TYITproject


,Rank,name,mcap,price,today,country
0,1,NVIDIANVDA,$3.763 T,$154.31,4.33%,USA
1,2,MicrosoftMSFT,$3.658 T,$492.27,0.44%,USA
2,3,AppleAAPL,$3.010 T,$201.56,0.63%,USA
3,4,AmazonAMZN,$2.250 T,$211.99,0.37%,USA
4,5,Alphabet (Google)GOOG,$2.075 T,$171.49,2.24%,USA
5,6,Meta Platforms (Facebook)META,$1.781 T,$708.68,0.49%,USA
6,7,Saudi Aramco2222.SR,$1.567 T,$6.48,0.66%,S. Arabia
7,8,BroadcomAVGO,$1.244 T,$264.65,0.33%,USA
8,9,TSMCTSM,$1.155 T,$222.74,1.20%,Taiwan
9,10,TeslaTSLA,$1.055 T,$327.55,3.79%,USA


In [2]:
def convert_to_number(val):
    val = val.replace('$', '').replace(',', '').strip()
    if 'T' in val:
        return float(val.replace('T', '')) * 1e12
    elif 'B' in val:
        return float(val.replace('B', '')) * 1e9
    elif 'e' in val:
        return float(val.replace(' ', ''))  # fix '3.763 e12'
    else:
        return float(val)

file["mcap"] = file["mcap"].apply(convert_to_number)



In [3]:
file
file.to_excel("cleaned_companies.xlsx",index=False)  # exporting data for  power bi 
file.to_excel(r"C:\Users\tusha\Desktop\cleaned_companies.xlsx",index=False)

In [4]:
file.rename(columns={
    "name":'Company name',
    "mcap":"Market cap",
    "price":"Stockprice",
    "today":"currently",
    "country":"Country",
},inplace=True)

In [5]:
file

,Rank,Company name,Market cap,Stockprice,currently,Country
0,1,NVIDIANVDA,3.763000e+12,$154.31,4.33%,USA
1,2,MicrosoftMSFT,3.658000e+12,$492.27,0.44%,USA
2,3,AppleAAPL,3.010000e+12,$201.56,0.63%,USA
3,4,AmazonAMZN,2.250000e+12,$211.99,0.37%,USA
4,5,Alphabet (Google)GOOG,2.075000e+12,$171.49,2.24%,USA
5,6,Meta Platforms (Facebook)META,1.781000e+12,$708.68,0.49%,USA
6,7,Saudi Aramco2222.SR,1.567000e+12,$6.48,0.66%,S. Arabia
7,8,BroadcomAVGO,1.244000e+12,$264.65,0.33%,USA
8,9,TSMCTSM,1.155000e+12,$222.74,1.20%,Taiwan
9,10,TeslaTSLA,1.055000e+12,$327.55,3.79%,USA


In [6]:
file.isnull().sum()

Rank            0
Company name    0
Market cap      0
Stockprice      0
currently       0
Country         0
dtype: int64

In [7]:
data=sqlite3.connect("sql.db")
file.to_sql("company_data",data,if_exists="replace",index=False)
std.read_sql_query("SELECT * FROM  company_data",data)
query =''' SELECT [Company name],[Market cap]FROM company_data ORDER  BY [Market cap] DESC LIMIT 5'''

In [8]:
table=data.execute("SELECT name FROM sqlite_master WHERE type='table';").fetchall()
print(table)
t1=std.read_sql_query("SELECT * FROM company_data LIMIT 20", data)
t1

[('job_seeker',), ('job placement',), ('job_placement',), ('job_switch_data',), ('salary_hike_experience',), ('company_data',)]


,Rank,Company name,Market cap,Stockprice,currently,Country
0,1,NVIDIANVDA,3.763000e+12,$154.31,4.33%,USA
1,2,MicrosoftMSFT,3.658000e+12,$492.27,0.44%,USA
2,3,AppleAAPL,3.010000e+12,$201.56,0.63%,USA
3,4,AmazonAMZN,2.250000e+12,$211.99,0.37%,USA
4,5,Alphabet (Google)GOOG,2.075000e+12,$171.49,2.24%,USA
5,6,Meta Platforms (Facebook)META,1.781000e+12,$708.68,0.49%,USA
6,7,Saudi Aramco2222.SR,1.567000e+12,$6.48,0.66%,S. Arabia
7,8,BroadcomAVGO,1.244000e+12,$264.65,0.33%,USA
8,9,TSMCTSM,1.155000e+12,$222.74,1.20%,Taiwan
9,10,TeslaTSLA,1.055000e+12,$327.55,3.79%,USA


In [9]:
# top 10 companies by market cap
query=''' SELECT [Company name],[Market cap] FROM company_data ORDER BY [Stockprice] DESC LIMIT 10 '''
std.read_sql_query(query,data)
file.to_excel("cleaned_companies.xlsx",index=False)

In [10]:
#
query=''' SELECT Country,SUM([Market cap]) AS Total_marketcap FROM company_data GROUP BY Country  ORDER BY Total_marketcap DESC LIMIT 6'''
std.read_sql_query(query,data)


,Country,Total_marketcap
0,USA,2.493709e+13
1,S. Arabia,1.567000e+12
2,Taiwan,1.155000e+12
3,China,5.928900e+11


In [11]:
#best performing company with roi 
# ipo prices
ipo_prices={
    "NVIDIANVDA":12,#
    "MicrosoftMSFT":22,# march 13,1986
	"AppleAAPL":22,#december 12,1980
	"AmazonAMZN":18,#may 15,1997	
	"Alphabet (Google)GOOG":85, #august19,2004
	"Meta Platforms (Facebook)META":38,#may 18,2012
    "Saudi Aramco2222.SR":8.53,
	"TeslaTSLA	  ":17,
	"VisaV":44,	
	"TencentTCEHY":13,	
	"OracleORCL":15,
	"NetflixNFLX"	:15 #2002
	
    
}

t1["IPO Price"]=t1["Company name"].map(ipo_prices)  # bhai ye use karna next time t1.column.str.strip() for removing spaces
t1.head(20)



,Rank,Company name,Market cap,Stockprice,currently,Country,IPO Price
0,1,NVIDIANVDA,3.763000e+12,$154.31,4.33%,USA,12.00
1,2,MicrosoftMSFT,3.658000e+12,$492.27,0.44%,USA,22.00
2,3,AppleAAPL,3.010000e+12,$201.56,0.63%,USA,22.00
3,4,AmazonAMZN,2.250000e+12,$211.99,0.37%,USA,18.00
4,5,Alphabet (Google)GOOG,2.075000e+12,$171.49,2.24%,USA,85.00
5,6,Meta Platforms (Facebook)META,1.781000e+12,$708.68,0.49%,USA,38.00
6,7,Saudi Aramco2222.SR,1.567000e+12,$6.48,0.66%,S. Arabia,8.53
7,8,BroadcomAVGO,1.244000e+12,$264.65,0.33%,USA,NaN
8,9,TSMCTSM,1.155000e+12,$222.74,1.20%,Taiwan,NaN
9,10,TeslaTSLA,1.055000e+12,$327.55,3.79%,USA,NaN


In [12]:
#removing dollor from stockprice coloumn
t1["Stockprice"]=t1["Stockprice"].replace('[/$,]','',regex=True).astype(float)
t1

,Rank,Company name,Market cap,Stockprice,currently,Country,IPO Price
0,1,NVIDIANVDA,3.763000e+12,154.31,4.33%,USA,12.00
1,2,MicrosoftMSFT,3.658000e+12,492.27,0.44%,USA,22.00
2,3,AppleAAPL,3.010000e+12,201.56,0.63%,USA,22.00
3,4,AmazonAMZN,2.250000e+12,211.99,0.37%,USA,18.00
4,5,Alphabet (Google)GOOG,2.075000e+12,171.49,2.24%,USA,85.00
5,6,Meta Platforms (Facebook)META,1.781000e+12,708.68,0.49%,USA,38.00
6,7,Saudi Aramco2222.SR,1.567000e+12,6.48,0.66%,S. Arabia,8.53
7,8,BroadcomAVGO,1.244000e+12,264.65,0.33%,USA,NaN
8,9,TSMCTSM,1.155000e+12,222.74,1.20%,Taiwan,NaN
9,10,TeslaTSLA,1.055000e+12,327.55,3.79%,USA,NaN


In [13]:
#calculating roi 
t1["ROI"]=((t1["Stockprice"]-t1["IPO Price"])/ t1["IPO Price"]) *100
t1.sort_values(by="ROI",ascending=False).head(5)

best_company=t1.sort_values(by="ROI", ascending=False).iloc[0]

t1["ROI"]=((t1["Stockprice"] - t1["IPO Price"]) / t1["IPO Price"]) * 100 

t1_Rank=t1.sort_values(by="ROI",ascending=False).head(20)

print(t1.columns.tolist());



['Rank', 'Company name', 'Market cap', 'Stockprice', 'currently', 'Country', 'IPO Price', 'ROI']


In [14]:
# best_roi=t1["ROI"].max()
# worst_roi=t1["ROI"].min()
# best_company=t1.loc[t1["ROI"].idxmax(),"Company name"]        its was for understanding and there was a better way to do it so i made new dataframe
# worst_company=t1.loc[t1["ROI"].idxmin(),"Company name"]
# print("Bes,t1["ROI"].max())
# print("worst_roi",t1["ROI"].min())

In [15]:
#Best and worst company ROI
best_company=t1.loc[t1["ROI"].idxmax(),["Company name","ROI"]]
worst_company=t1.loc[t1["ROI"].idxmin(),["Company name","ROI"]]


performance_card=std.DataFrame([best_company,worst_company])
performance_card.index=["best performer","worst performer"]
performance_card

,Company name,ROI
best performer,NetflixNFLX,8400.000000
worst performer,Saudi Aramco2222.SR,-24.032825


In [16]:
#voltility card
import numpy as np
#data here is random in future this project can also use yahoo for calulating real time data 

np.random.seed(42)
t1["volatility"]=np.random.uniform(5,30,size=len(t1))
#card
print(t1.columns)

#higest risk company
voltaility_card=t1[["Company name","volatility"]].copy()
voltaility_card.sort_values(by="volatility",ascending=False,inplace=True)
voltaility_card.head()
voltaility_card.tail()


Index(['Rank', 'Company name', 'Market cap', 'Stockprice', 'currently',
       'Country', 'IPO Price', 'ROI', 'volatility'],
      dtype='object')


,Company name,volatility
14,VisaV,9.545624
4,Alphabet (Google)GOOG,8.900466
5,Meta Platforms (Facebook)META,8.899863
6,Saudi Aramco2222.SR,6.452090
10,Berkshire Hathaway BRK-B,5.514612


In [17]:
# giving a structure so data can be easily imported
t1.to_excel("cleaned_companies.xlsx",index=False)
print("success")

success


In [18]:
#creating small table for power bi so its easy
best=t1.loc[t1["ROI"].idxmax(),
["Company name","ROI"]]
worst=t1.loc[t1["ROI"].idxmin(),
["Company name","ROI"]]
print(worst);

kpi_card=std.DataFrame({
    "best company":[best["Company name"]],
    "best roi":[best["ROI"]],
    "worst company":[worst["Company name"]],
    "worst roi":[best["ROI"]],
    "average roi":[t1["ROI"].mean()]
})
#save to excel
kpi_card.to_excel("kpi_summary.xlsx",index=False)
print("File saved as kpi_card.xlsx")

Company name    Saudi Aramco2222.SR
ROI                      -24.032825
Name: 6, dtype: object
File saved as kpi_card.xlsx


In [19]:
data=sqlite3.connect("sql.db")
file.to_sql("company_data",data,if_exists="replace",index=False)
std.read_sql_query("SELECT * FROM  company_data",data)

,Rank,Company name,Market cap,Stockprice,currently,Country
0,1,NVIDIANVDA,3.763000e+12,$154.31,4.33%,USA
1,2,MicrosoftMSFT,3.658000e+12,$492.27,0.44%,USA
2,3,AppleAAPL,3.010000e+12,$201.56,0.63%,USA
3,4,AmazonAMZN,2.250000e+12,$211.99,0.37%,USA
4,5,Alphabet (Google)GOOG,2.075000e+12,$171.49,2.24%,USA
5,6,Meta Platforms (Facebook)META,1.781000e+12,$708.68,0.49%,USA
6,7,Saudi Aramco2222.SR,1.567000e+12,$6.48,0.66%,S. Arabia
7,8,BroadcomAVGO,1.244000e+12,$264.65,0.33%,USA
8,9,TSMCTSM,1.155000e+12,$222.74,1.20%,Taiwan
9,10,TeslaTSLA,1.055000e+12,$327.55,3.79%,USA


In [20]:
# import pandas as pd
# import sqlite3

# # Assume df is your cleaned dataframe

# # 1️⃣ Split company name + ticker
# file[["Company_Name", "Ticker"]] = file["Company name"].str.extract(r"(.+?)([A-Z\.]+)$")

# # 2️⃣ Clean stock price (remove $)
# file["Stockprice"] = file["Stockprice"].replace(r"[$,]", "", regex=True).astype(float)

# # 3️⃣ Clean percentage column (remove %)
# file["currently"] = file["currently"].str.replace("%", "").astype(float)

# # 4️⃣ Rename columns cleanly
# file = file.rename(columns={
#     "Rank": "Rank",
#     "Market cap": "Market_Cap",
#     "Stockprice": "Stock_Price",
#     "currently": "Daily_Change_Percent",
#     "Country": "Country"
# })

# # 5️⃣ Drop old messy column
# file = file.drop(columns=["Company name"])

# # 6️⃣ Save clean version to SQL
# conn = sqlite3.connect("sql.db")


# conn.close()




In [21]:
std.read_sql_query("SELECT * FROM  company_data",data)
file["Ticker"] = file["Company name"].str.extract(r'([A-Z0-9\.]{2,10})$')


In [22]:
file["Company_Name"] = file["Company name"].str.replace(r'([A-Z0-9\.]{2,10})$', '', regex=True).str.strip()

In [23]:
file[["Company name", "Company_Name", "Ticker"]].head(20)

,Company name,Company_Name,Ticker
0,NVIDIANVDA,,NVIDIANVDA
1,MicrosoftMSFT,Microsoft,MSFT
2,AppleAAPL,Apple,AAPL
3,AmazonAMZN,Amazon,AMZN
4,Alphabet (Google)GOOG,Alphabet (Google),GOOG
5,Meta Platforms (Facebook)META,Meta Platforms (Facebook),META
6,Saudi Aramco2222.SR,Saudi Aramco,2222.SR
7,BroadcomAVGO,Broadcom,AVGO
8,TSMCTSM,,TSMCTSM
9,TeslaTSLA,Tesla,TSLA


In [24]:
file = file.drop(columns=["Company_Name", "Ticker"], errors="ignore")

In [25]:
import re
import pandas as pd

def split_company(text):
    text = text.strip()
    
    # Match ticker at end (1–5 uppercase letters, or with dot/dash)
    match = re.search(r'([A-Z]{1,5}(\.[A-Z]{1,5}|-[A-Z]{1,5})?)$', text)
    
    if match:
        ticker = match.group(1)
        name = text[:-len(ticker)].strip()
        return pd.Series([name, ticker])
    else:
        return pd.Series([text, None])

file[["Company_Name", "Ticker"]] = file["Company name"].apply(split_company)

file[["Company name", "Company_Name", "Ticker"]].head(20)

,Company name,Company_Name,Ticker
0,NVIDIANVDA,NVIDI,ANVDA
1,MicrosoftMSFT,Microsoft,MSFT
2,AppleAAPL,Apple,AAPL
3,AmazonAMZN,Amazon,AMZN
4,Alphabet (Google)GOOG,Alphabet (Google),GOOG
5,Meta Platforms (Facebook)META,Meta Platforms (Facebook),META
6,Saudi Aramco2222.SR,Saudi Aramco2222.,SR
7,BroadcomAVGO,Broadcom,AVGO
8,TSMCTSM,TS,MCTSM
9,TeslaTSLA,Tesla,TSLA


In [26]:
file.loc[file["Company name"] == "NVIDIANVDA", ["Company_Name", "Ticker"]] = ["NVIDIA", "NVDA"]

In [27]:
file.loc[file["Company name"] == "Saudi Aramco2222.SR", ["Company_Name", "Ticker"]] = ["Saudi Aramco", "2222.SR"]

In [28]:
file.loc[file["Company name"] == "TSMCTSM", ["Company_Name", "Ticker"]] = ["Taiwan Semiconductor", "TSM"]

In [29]:
file[["Company name", "Company_Name", "Ticker"]]

,Company name,Company_Name,Ticker
0,NVIDIANVDA,NVIDIA,NVDA
1,MicrosoftMSFT,Microsoft,MSFT
2,AppleAAPL,Apple,AAPL
3,AmazonAMZN,Amazon,AMZN
4,Alphabet (Google)GOOG,Alphabet (Google),GOOG
5,Meta Platforms (Facebook)META,Meta Platforms (Facebook),META
6,Saudi Aramco2222.SR,Saudi Aramco,2222.SR
7,BroadcomAVGO,Broadcom,AVGO
8,TSMCTSM,Taiwan Semiconductor,TSM
9,TeslaTSLA,Tesla,TSLA


In [30]:
file = file[~file["Company name"].str.contains("Saudi Aramco", na=False)]

In [31]:
file = file.reset_index(drop=True)

In [32]:
file[["Company name", "Company_Name", "Ticker"]].head(19)

,Company name,Company_Name,Ticker
0,NVIDIANVDA,NVIDIA,NVDA
1,MicrosoftMSFT,Microsoft,MSFT
2,AppleAAPL,Apple,AAPL
3,AmazonAMZN,Amazon,AMZN
4,Alphabet (Google)GOOG,Alphabet (Google),GOOG
5,Meta Platforms (Facebook)META,Meta Platforms (Facebook),META
6,BroadcomAVGO,Broadcom,AVGO
7,TSMCTSM,Taiwan Semiconductor,TSM
8,TeslaTSLA,Tesla,TSLA
9,Berkshire Hathaway BRK-B,Berkshire Hathaway,BRK-B


In [33]:
conn = sqlite3.connect("sql.db")
file.to_sql("company_data", conn, if_exists="replace", index=False)
conn.close()

In [34]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("sql.db")

check_df = pd.read_sql_query("SELECT * FROM company_data", conn)

conn.close()

check_df.head(15)

,Rank,Company name,Market cap,Stockprice,currently,Country,Company_Name,Ticker
0,1,NVIDIANVDA,3.763000e+12,$154.31,4.33%,USA,NVIDIA,NVDA
1,2,MicrosoftMSFT,3.658000e+12,$492.27,0.44%,USA,Microsoft,MSFT
2,3,AppleAAPL,3.010000e+12,$201.56,0.63%,USA,Apple,AAPL
3,4,AmazonAMZN,2.250000e+12,$211.99,0.37%,USA,Amazon,AMZN
4,5,Alphabet (Google)GOOG,2.075000e+12,$171.49,2.24%,USA,Alphabet (Google),GOOG
5,6,Meta Platforms (Facebook)META,1.781000e+12,$708.68,0.49%,USA,Meta Platforms (Facebook),META
6,8,BroadcomAVGO,1.244000e+12,$264.65,0.33%,USA,Broadcom,AVGO
7,9,TSMCTSM,1.155000e+12,$222.74,1.20%,Taiwan,Taiwan Semiconductor,TSM
8,10,TeslaTSLA,1.055000e+12,$327.55,3.79%,USA,Tesla,TSLA
9,11,Berkshire Hathaway BRK-B,1.049000e+12,$486.21,1.47%,USA,Berkshire Hathaway,BRK-B


In [35]:
check_df.dtypes

Rank              int64
Company name     object
Market cap      float64
Stockprice       object
currently        object
Country          object
Company_Name     object
Ticker           object
dtype: object

In [36]:
file["Stockprice"] = (
    file["Stockprice"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

file["currently"] = (
    file["currently"]
    .str.replace("%", "", regex=False)
    .astype(float)
)

In [37]:
conn = sqlite3.connect("sql.db")
file.to_sql("company_data", conn, if_exists="replace", index=False)
conn.close()

In [38]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("sql.db")

check_df = pd.read_sql_query("SELECT * FROM company_data", conn)

conn.close()

check_df.head(15)

,Rank,Company name,Market cap,Stockprice,currently,Country,Company_Name,Ticker
0,1,NVIDIANVDA,3.763000e+12,154.31,4.33,USA,NVIDIA,NVDA
1,2,MicrosoftMSFT,3.658000e+12,492.27,0.44,USA,Microsoft,MSFT
2,3,AppleAAPL,3.010000e+12,201.56,0.63,USA,Apple,AAPL
3,4,AmazonAMZN,2.250000e+12,211.99,0.37,USA,Amazon,AMZN
4,5,Alphabet (Google)GOOG,2.075000e+12,171.49,2.24,USA,Alphabet (Google),GOOG
5,6,Meta Platforms (Facebook)META,1.781000e+12,708.68,0.49,USA,Meta Platforms (Facebook),META
6,8,BroadcomAVGO,1.244000e+12,264.65,0.33,USA,Broadcom,AVGO
7,9,TSMCTSM,1.155000e+12,222.74,1.20,Taiwan,Taiwan Semiconductor,TSM
8,10,TeslaTSLA,1.055000e+12,327.55,3.79,USA,Tesla,TSLA
9,11,Berkshire Hathaway BRK-B,1.049000e+12,486.21,1.47,USA,Berkshire Hathaway,BRK-B


In [39]:
check_df.dtypes
check_df.isnull().sum()

Rank            0
Company name    0
Market cap      0
Stockprice      0
currently       0
Country         0
Company_Name    0
Ticker          0
dtype: int64

In [40]:
df = check_df.copy()

# 1 Total Market Cap
total_market_cap = df["Market cap"].sum()

# 2 Largest Company
largest_row = df.loc[df["Market cap"].idxmax()]
largest_company = largest_row["Company_Name"]
largest_cap = largest_row["Market cap"]

# 3 Highest Daily Gainer
top_gainer_row = df.loc[df["currently"].idxmax()]
top_gainer = top_gainer_row["Company_Name"]
top_gain_percent = top_gainer_row["currently"]

# 4 Average Daily Movement
avg_daily_move = df["currently"].mean()

print("Total Market Cap:", total_market_cap)
print("Largest Company:", largest_company)
print("Top Gainer:", top_gainer)
print("Average Daily Move:", avg_daily_move)

Total Market Cap: 26684980000000.0
Largest Company: NVIDIA
Top Gainer: NVIDIA
Average Daily Move: 1.3373684210526315


In [41]:
total_market_cap_trillion = total_market_cap / 1e12
print("Total Market Cap (Trillion $):", round(total_market_cap_trillion, 2))

Total Market Cap (Trillion $): 26.68


In [42]:
import requests
import pandas as pd

api_key = "kM8pZmDinPtySSyzBweAcZCEsxzoiaQF"

ticker = "AAPL"  # test with one company first

url = f"https://financialmodelingprep.com/stable/quote?symbol={ticker}&apikey={api_key}"

response = requests.get(url)

data = response.json()

data

[{'symbol': 'AAPL',
  'name': 'Apple Inc.',
  'price': 253.79,
  'changePercentage': 2.90313,
  'change': 7.16,
  'volume': 48770963,
  'dayLow': 247.101,
  'dayHigh': 255.44,
  'yearHigh': 288.62,
  'yearLow': 169.21,
  'marketCap': 3730186664919.0005,
  'priceAvg50': 260.0538,
  'priceAvg200': 248.29205,
  'exchange': 'NASDAQ',
  'open': 247.91,
  'previousClose': 246.63,
  'timestamp': 1774987202}]